In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import neighbors
from sklearn import metrics
from sklearn import ensemble

In [4]:
data = pd.read_csv('preprocessed_data.csv', index_col=0)

In [5]:
data.head()

,Bachelor's,Intermediate,Master's,Matric,PhD,Finance,Science,Business,Computer Science,Arts,...,Marketing,Tally ERP,AWS Certified,Mental Health Basics,Digital Marketing,CFA Level 1,Creative Writing,Google Data Analytics,CGPA/Percentage,Recommended Career
0,1,0,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,67,Business Analyst
1,0,1,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,67,Software Engineer
2,0,0,1,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,90,Financial Analyst
3,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,75,Clerk
4,0,0,0,1,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,83,Sales Assistant


In [6]:
target = 'Recommended Career'

In [7]:
x = data.drop(target, axis=1)

In [8]:
target_names = data[target].unique()

y = data[target]

label_encoder = preprocessing.LabelEncoder()
y_encoded = label_encoder.fit_transform( data[target] )

y_encoded

array([ 0, 11,  3, ...,  3, 11,  1], shape=(5000,))

In [9]:
x_trainval, x_test, y_trainval, y_test = model_selection.train_test_split(x, y_encoded, test_size=0.33, stratify=y, random_state=7)

In [10]:
x_train, x_val, y_train, y_val = model_selection.train_test_split(x_trainval, y_trainval, test_size=0.2, stratify=y_trainval, random_state=7)

In [11]:
scaler = preprocessing.StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)

In [12]:
scaler = preprocessing.StandardScaler()
scaler.fit(x_trainval)
x_trainval = scaler.transform(x_trainval)
x_test = scaler.transform(x_test)

In [13]:
def train_and_evaluate(model, x_train, y_train, x_val, y_val):
    model.fit(x_train, y_train)

    return model.score(x_val, y_val)

In [14]:
ks = np.arange(5,30, 5)
ks

array([ 5, 10, 15, 20, 25])

In [15]:
optimal_k = 0
optimal_accuracy = 0

for k in ks:
    model = neighbors.KNeighborsClassifier(n_neighbors=k, metric='cosine')
    score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
    if score > optimal_accuracy:
        optimal_accuracy = score
        optimal_k = k

print(optimal_k, optimal_accuracy)

15 0.1044776119402985


In [16]:
ks = np.arange(optimal_k-4, optimal_k+4, 1)
ks

array([11, 12, 13, 14, 15, 16, 17, 18])

In [17]:
optimal_accuracy = 0

for k in ks:
    model = neighbors.KNeighborsClassifier(n_neighbors=k, metric='cosine')
    score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
    if score > optimal_accuracy:
        optimal_accuracy = score
        optimal_k = k

print(optimal_k, optimal_accuracy)

15 0.1044776119402985


In [18]:
model = neighbors.KNeighborsClassifier(n_neighbors=optimal_k, metric='cosine')

In [19]:
model.fit(x_trainval, y_trainval)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",np.int64(15)
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [20]:
model.score(x_train, y_train)

0.21641791044776118

In [21]:
model.score(x_test, y_test)

0.07212121212121213

In [22]:
y_test_predicted = model.predict(x_test)

In [23]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.07      0.12      0.09       130
           1       0.10      0.16      0.12       143
           2       0.04      0.04      0.04       136
           3       0.07      0.07      0.07       136
           4       0.07      0.07      0.07       136
           5       0.08      0.08      0.08       142
           6       0.05      0.03      0.04       143
           7       0.07      0.06      0.07       134
           8       0.09      0.08      0.08       138
           9       0.06      0.04      0.05       136
          10       0.09      0.07      0.07       138
          11       0.07      0.04      0.05       138

    accuracy                           0.07      1650
   macro avg       0.07      0.07      0.07      1650
weighted avg       0.07      0.07      0.07      1650



In [24]:
cmatrix = metrics.confusion_matrix(y_test, y_test_predicted)
cmatrix

array([[15, 15, 14, 18, 14,  6,  5, 16,  7,  6,  8,  6],
       [18, 23,  4,  9, 12, 16,  7, 10, 14,  9, 10, 11],
       [24, 28,  6, 11, 13, 16,  5,  7,  9,  8,  3,  6],
       [16, 19, 19,  9,  6, 16,  9,  8,  9,  9,  8,  8],
       [19, 21, 15, 10,  9, 17,  9,  8,  8,  6,  7,  7],
       [13, 19, 10, 15, 13, 12,  9, 12, 14,  7,  9,  9],
       [23, 26, 23,  9, 10,  8,  5,  8,  9, 10,  9,  3],
       [11, 15, 13, 12,  8, 16, 14,  8,  7, 13, 12,  5],
       [19, 18,  5, 15, 14, 11, 12, 10, 11,  5,  9,  9],
       [19, 17, 22,  8,  8, 11, 11,  8, 11,  6, 10,  5],
       [17, 20, 14,  8,  8, 11, 12,  3, 17,  7,  9, 12],
       [14, 17, 15, 10, 10, 19, 10, 11, 10,  7,  9,  6]])

In [25]:
ks = np.arange(5,30, 5)
n_estimators = np.arange(10, 110, 10)

In [26]:
optimal_k = 0
optimal_estimators = 0
optimal_accuracy = 0

for k in ks:
    for n_estimator in n_estimators:
        base_model = neighbors.KNeighborsClassifier(n_neighbors=k, metric='cosine')
        model = ensemble.BaggingClassifier(base_model, n_estimators=n_estimator, max_samples=0.9, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_estimators = n_estimator
            optimal_k = k
            
print(optimal_k, optimal_estimators, optimal_accuracy)

15 30 0.11940298507462686


In [28]:
base_model = neighbors.KNeighborsClassifier(n_neighbors=optimal_k, metric='cosine')
model = ensemble.BaggingClassifier(base_model, n_estimators=optimal_estimators)

In [29]:
model.fit(x_trainval, y_trainval)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",KNeighborsCla...=np.int64(15))
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",np.int64(30)
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [30]:
model.score(x_test, y_test)

0.07333333333333333

In [31]:
y_test_predicted = model.predict(x_test)

In [32]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.08      0.07      0.07       130
           1       0.07      0.07      0.07       143
           2       0.05      0.04      0.05       136
           3       0.09      0.09      0.09       136
           4       0.08      0.07      0.07       136
           5       0.07      0.08      0.08       142
           6       0.08      0.08      0.08       143
           7       0.06      0.06      0.06       134
           8       0.08      0.10      0.09       138
           9       0.06      0.06      0.06       136
          10       0.08      0.09      0.09       138
          11       0.07      0.07      0.07       138

    accuracy                           0.07      1650
   macro avg       0.07      0.07      0.07      1650
weighted avg       0.07      0.07      0.07      1650



In [33]:
cmatrix = metrics.confusion_matrix(y_test, y_test_predicted)
cmatrix

array([[ 9,  5,  9, 16, 12,  8,  6, 17, 12, 13, 11, 12],
       [ 9, 10,  4, 14, 12, 13, 11, 15, 17, 11, 13, 14],
       [13, 14,  6, 14, 15, 16,  8,  8, 10, 12,  8, 12],
       [11, 14, 15, 12,  8, 12, 10, 12, 12,  8, 11, 11],
       [13, 13, 13, 11,  9, 18, 11,  8,  9,  8, 11, 12],
       [ 6, 14, 10, 11, 11, 12, 10, 12, 17, 10, 12, 17],
       [ 7, 16, 16, 10, 11, 20, 11,  8, 13, 13, 13,  5],
       [ 5,  7, 11, 11,  7, 16, 11,  8, 15, 14, 19, 10],
       [12, 18,  6, 12,  8, 10, 15, 10, 14,  8, 13, 12],
       [13, 10, 15,  9,  6, 12, 15, 10, 14,  8, 15,  9],
       [11, 12,  9,  6,  7, 10, 15,  6, 22, 12, 13, 15],
       [ 7,  9, 13, 10, 10, 15, 11, 15, 16,  9, 14,  9]])